### Import required libraries.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### DATA READING

In [2]:
fog_star = pd.read_csv("sensor_data.csv")

In [3]:
fog_star= fog_star[fog_star.activity>0]

In [4]:
fog_star.columns

Index(['timestamp', 'ankleL_acc_x', 'ankleL_acc_y', 'ankleL_acc_z',
       'ankleL_gyro_x', 'ankleL_gyro_y', 'ankleL_gyro_z', 'ankleR_acc_x',
       'ankleR_acc_y', 'ankleR_acc_z', 'ankleR_gyro_x', 'ankleR_gyro_y',
       'ankleR_gyro_z', 'back_acc_x', 'back_acc_y', 'back_acc_z',
       'back_gyro_x', 'back_gyro_y', 'back_gyro_z', 'wrist_acc_x',
       'wrist_acc_y', 'wrist_acc_z', 'wrist_gyro_x', 'wrist_gyro_y',
       'wrist_gyro_z', 'activity', 'fog', 'fog_severity', 'subjectID',
       'sessionID', 'taskID'],
      dtype='object')

In [5]:
fog_star

,timestamp,ankleL_acc_x,ankleL_acc_y,ankleL_acc_z,ankleL_gyro_x,ankleL_gyro_y,ankleL_gyro_z,ankleR_acc_x,ankleR_acc_y,ankleR_acc_z,...,wrist_acc_z,wrist_gyro_x,wrist_gyro_y,wrist_gyro_z,activity,fog,fog_severity,subjectID,sessionID,taskID
0,0.000000,-1.014984,0.133386,0.024818,0.212810,1.589006,-0.167686,-1.005514,-0.097270,0.013287,...,0.428789,0.592055,-0.191380,0.327340,2,0,0,1,1,1
1,0.016667,-1.000777,0.136812,0.034778,-0.074904,1.756708,-0.114430,-1.007110,-0.095888,0.006187,...,0.428819,0.734625,-0.414149,0.717459,2,0,0,1,1,1
2,0.033333,-1.005732,0.130560,0.032259,0.183994,1.572684,-0.083382,-1.004861,-0.099395,0.007403,...,0.413886,0.426720,-0.243212,0.300301,2,0,0,1,1,1
3,0.050000,-1.003626,0.129369,0.035319,-0.066717,1.803254,-0.083298,-1.003089,-0.098310,0.010602,...,0.418330,0.759547,-0.233320,0.525266,2,0,0,1,1,1
4,0.066667,-1.004712,0.140318,0.029176,0.020656,1.400361,-0.203291,-1.001985,-0.098686,0.012207,...,0.431261,0.676813,-0.193347,0.466261,2,0,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329012,5483.533333,NaN,NaN,NaN,NaN,NaN,NaN,-0.993790,0.068745,-0.087219,...,0.495556,9.315055,5.190101,0.526538,3,0,0,9,1,7
329013,5483.550000,NaN,NaN,NaN,NaN,NaN,NaN,-0.992370,0.084500,-0.096502,...,0.500704,7.553753,5.093344,0.790108,3,0,0,9,1,7
329014,5483.566667,NaN,NaN,NaN,NaN,NaN,NaN,-0.994886,0.073666,-0.112879,...,0.497469,2.824992,4.896034,1.201217,3,0,0,9,1,7
329015,5483.583333,NaN,NaN,NaN,NaN,NaN,NaN,-0.988763,0.075161,-0.097260,...,0.500338,-2.555501,4.621429,0.795947,3,0,0,9,1,7


### DATA PLOTS AND ANALYTICS

In [6]:
# ======= fogstar_paper_figs_like_examples.py =======
# Usage:
#   outs = make_fogstar_example_style_figures(fog_star, outdir="paper_figures")
#   print(outs)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# --- Mapping dictionaries ---
ACTIVITY_MAP = {
    1: "Walking",
    2: "Sit",
    3: "Stand",
    4: "Sit-to-Stand",
    5: "Stand-to-Sit",
    6: "Turn Right",
    7: "Turn Left"
}

FOG_SEVERITY_MAP = {
    1: "Shuffling (small steps)",
    2: "Trembling (knees rapid)",
    3: "Complete akinesia"
}

TASK_MAP = {
    1: "Timed Up-and-Go",
    2: "Stand 1 min",
    3: "Walk back/forth",
    4: "Walk + doorway",
    5: "Walk + carry water",
    6: "Walk + count backward",
    7: "360° turn"
}


# ---------- helpers ----------
def _ensure_datetime(ts: pd.Series) -> pd.Series:
    s = pd.Series(ts)
    if np.issubdtype(s.dtype, np.datetime64):
        return pd.to_datetime(s)
    if np.issubdtype(s.dtype, np.number):
        s = s.astype("int64")
        med = int(np.median(s))
        if med > 10**17: return pd.to_datetime(s, unit="ns")
        if med > 10**11: return pd.to_datetime(s, unit="ms")
        return pd.to_datetime(s, unit="s")
    return pd.to_datetime(s, errors="coerce")

def _infer_fs(dt: pd.Series) -> float | None:
    try:
        d = np.diff(np.sort(dt.astype("int64"))) / 1e9
        d = d[(d > 0) & np.isfinite(d)]
        if d.size == 0: return None
        return 1.0 / np.median(d)
    except Exception:
        return None

def _durations_by_group(df: pd.DataFrame, by_cols, fs: float = 60.0) -> pd.DataFrame:
    """
    Total minutes per group; sums over sessions so multiple sessions per subject
    are accumulated. Uses timestamps when near-regular; otherwise uses counts/fs.
    Keeps natural order of group IDs.
    """
    d = df.copy()
    d['timestamp'] = _ensure_datetime(d['timestamp'])
    d = d.dropna(subset=['timestamp'])

    inferred = _infer_fs(d['timestamp'])
    use_counts = inferred is None or not (0.9*fs <= inferred <= 1.1*fs)

    group_keys = list(by_cols)
    group_plus_session = group_keys + (['sessionID'] if 'sessionID' in d.columns and 'sessionID' not in group_keys else [])

    if not use_counts:
        span = (d.groupby(group_plus_session)['timestamp']
                  .agg(['min','max'])
                  .reset_index())
        span['minutes'] = (span['max'] - span['min']).dt.total_seconds()/60.0
    else:
        counts = d.groupby(group_plus_session)['timestamp'].size().reset_index(name='n')
        counts['minutes'] = counts['n'] / fs / 60.0
        span = counts

    out = span.groupby(by_cols, sort=False)['minutes'].sum().reset_index()
    # Ensure categories like Subject 1, Subject 2 remain in natural sorted order if strings
    for col in by_cols:
        if out[col].dtype == 'object':
            out[col] = pd.Categorical(out[col], categories=sorted(out[col].unique(), key=str))
    return out


def _style():
    plt.rcParams.update({
        "font.size": 13,
        "axes.titlesize": 16,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.dpi": 150,
        "savefig.dpi": 300
    })

def _annotate_bars(ax, bars):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h,
                f"{h:.0f}", ha='center', va='bottom', fontsize=12)

# ---------- Figure A: three-panel bar charts ----------
def _figure_three_panels(sub_df, task_df, act_df, outpath):
    _style()
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

    # Subjects
    ax = axes[0]
    bars = ax.bar(sub_df.iloc[:,0].astype(str), sub_df['minutes'].values, color="#a6d8f0")  # light sky blue
    ax.set_title("Recording Time per Subject")
    ax.set_xlabel("Subject")
    ax.set_ylabel("Total Duration (min)")
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    _annotate_bars(ax, bars)

    # Tasks
    ax = axes[1]
    bars = ax.bar(task_df.iloc[:,0].astype(str), task_df['minutes'].values, color="#f2a6a6") # light salmon
    ax.set_title("Recording Time by Task")
    ax.set_xlabel("Task")
    ax.set_ylabel("Total Duration (min)")
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    _annotate_bars(ax, bars)

    # Activities
    ax = axes[2]
    bars = ax.bar(act_df.iloc[:,0].astype(str), act_df['minutes'].values, color="#bfe8b5")   # light green
    ax.set_title("Time per Activity Class")
    ax.set_xlabel("Activity Class")
    ax.set_ylabel("Total Duration (min)")
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    _annotate_bars(ax, bars)

    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

# ---------- Figure B: combined sectioned bar chart ----------
def _figure_combined(sub_df, task_df, act_df, outpath):
    _style()
    # Build combined x & y with separators
    labels_sub = sub_df.iloc[:,0].astype(str).tolist()
    y_sub     = sub_df['minutes'].round(0).tolist()

    labels_task = task_df.iloc[:,0].astype(str).tolist()
    y_task      = task_df['minutes'].round(0).tolist()

    labels_act = act_df.iloc[:,0].astype(str).tolist()
    y_act      = act_df['minutes'].round(0).tolist()

    labels = labels_sub + labels_task + labels_act
    yvals  = y_sub + y_task + y_act

    colors = (["#a6d8f0"] * len(labels_sub) +
              ["#f2a6a6"] * len(labels_task) +
              ["#bfe8b5"] * len(labels_act))

    fig, ax = plt.subplots(figsize=(24, 16))
    ax.set_title("Recording Time Analysis - All Categories Combined", pad=18)

    x = np.arange(len(labels))
    bars = ax.bar(x, yvals, color=colors, edgecolor="white", linewidth=0.6)

    # Add value labels
    for i, rect in enumerate(bars):
        ax.text(rect.get_x()+rect.get_width()/2, rect.get_height(),
                f"{yvals[i]:.0f}", ha="center", va="bottom", fontsize=11)

    # X ticks
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_xlabel("Categories")
    ax.set_ylabel("Total Duration (min)")
    ax.grid(axis='y', linestyle='--', linewidth=0.7, alpha=0.5)
    ax.set_axisbelow(True)

    # Vertical dashed separators between sections
    split1 = len(labels_sub) - 0.5
    split2 = len(labels_sub) + len(labels_task) - 0.5
    ymax = max(yvals) * 1.15
    for xpos in [split1, split2]:
        ax.axvline(x=xpos, linestyle='--', linewidth=1.2, alpha=0.5)

    # Section headers above bars (blue/red/green like your example)
    ax.text(len(labels_sub)/2 - 0.5, ymax, "SUBJECTS", ha='center', va='bottom', fontsize=14, color="#3b73ff")
    ax.text(len(labels_sub) + len(labels_task)/2 - 0.5, ymax, "TASKS", ha='center', va='bottom', fontsize=14, color="#ff4d4d")
    ax.text(len(labels) - len(labels_act)/2 - 0.5, ymax, "ACTIVITIES", ha='center', va='bottom', fontsize=14, color="#38a169")

    ax.set_ylim(0, ymax)

    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

# ---------- public function ----------
def make_fogstar_example_style_figures(df: pd.DataFrame, outdir="paper_figures", text_labels: bool = True, fs: float = 60.0):
    """
    Create images closely matching the provided examples:
      - 'triptych_bars.png' : three aligned bar plots
      - 'combined_block_bars.png' : one chart with three colored sections
    Returns dict of output paths.
    """
    os.makedirs(outdir, exist_ok=True)

    # Compute minutes per group (sorted desc)
    sub_df  = _durations_by_group(df, ['subjectID'], fs=fs)      if 'subjectID' in df.columns else pd.DataFrame({'subjectID':[], 'minutes':[]})
    task_df = _durations_by_group(df, ['taskID'], fs=fs)          if 'taskID' in df.columns else pd.DataFrame({'taskID':[], 'minutes':[]})
    act_df  = _durations_by_group(df, ['activity'], fs=fs)        if 'activity' in df.columns else pd.DataFrame({'activity':[], 'minutes':[]})

    if (text_labels):
        # Compute minutes per group (without sorting by value)
        sub_df  = _durations_by_group(df, ['subjectID'], fs=fs)

        task_df = _durations_by_group(df, ['taskID'], fs=fs)
        if not task_df.empty:
            task_df['taskID'] = task_df['taskID'].map(TASK_MAP).fillna(task_df['taskID'])

        act_df  = _durations_by_group(df, ['activity'], fs=fs)
        if not act_df.empty:
            act_df['activity'] = act_df['activity'].map(ACTIVITY_MAP).fillna(act_df['activity'])

    # Ensure there is at least something to plot
    if any(len(d)==0 for d in [sub_df, task_df, act_df]):
        raise ValueError("Need non-empty 'subjectID', 'taskID', and 'activity' groupings to reproduce the figures.")

    outA = os.path.join(outdir, "triptych_bars.png")
    _figure_three_panels(sub_df, task_df, act_df, outA)

    outB = os.path.join(outdir, "combined_block_bars.png")
    _figure_combined(sub_df, task_df, act_df, outB)

    return {"triptych": outA, "combined": outB}



In [7]:
outs = make_fogstar_example_style_figures(fog_star, outdir="paper_figures", text_labels=True)
print(outs)


{'triptych': 'paper_figures/triptych_bars.png', 'combined': 'paper_figures/combined_block_bars.png'}


In [ ]:
# ======= fogstar_fog_distributions.py =======
# Optional labels (used for pretty x-ticks)
FOG_LABELS = {0: "Non-FOG", 1: "FOG"}
FOG_SEVERITY_MAP = {
    1: "Shuffling",
    2: "Trembling",
    3: "Akinesia"
}

def _style():
    plt.rcParams.update({
        "font.size": 13,
        "axes.titlesize": 16,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.dpi": 150,
        "savefig.dpi": 300
    })

def _annotate_bars(ax, bars):
    for r in bars:
        h = r.get_height()
        ax.text(r.get_x()+r.get_width()/2, h, f"{h:.0f}", ha='center', va='bottom', fontsize=12)

def _triptych(axs, xlabels, samples, seconds, minutes, title_prefix, colors):
    # left: samples
    bars = axs[0].bar(xlabels, samples, color=colors[0])
    axs[0].set_title(f"{title_prefix}: Samples")
    axs[0].set_ylabel("Count")
    axs[0].grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    _annotate_bars(axs[0], bars)
    axs[0].tick_params(axis='x', rotation=20)

    # middle: seconds
    bars = axs[1].bar(xlabels, seconds, color=colors[1])
    axs[1].set_title(f"{title_prefix}: Seconds")
    axs[1].set_ylabel("s")
    axs[1].grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    _annotate_bars(axs[1], bars)
    axs[1].tick_params(axis='x', rotation=20)

    # right: minutes
    bars = axs[2].bar(xlabels, minutes, color=colors[2])
    axs[2].set_title(f"{title_prefix}: Minutes")
    axs[2].set_ylabel("min")
    axs[2].grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    _annotate_bars(axs[2], bars)
    axs[2].tick_params(axis='x', rotation=20)

def make_fog_distribution_figures(
    df: pd.DataFrame,
    outdir: str = "paper_figures",
    fs: float = 60.0
) -> dict:
    """
    Creates:
      - fog_triptych.png           (FOG vs Non-FOG: Samples/Seconds/Minutes)
      - fog_severity_triptych.png  (Severity 1..3: Samples/Seconds/Minutes)
    Returns a dict with file paths and numeric summaries.
    """
    os.makedirs(outdir, exist_ok=True)
    _style()

    # ---------- FOG vs Non-FOG ----------
    if 'fog' not in df.columns:
        raise ValueError("DataFrame must contain a 'fog' column.")

    # Fixed order: Non-FOG (0), FOG (1)
    fog_order = [0, 1]
    fog_counts = df['fog'].value_counts().reindex(fog_order).fillna(0).astype(int)
    fog_samples = fog_counts.values
    fog_seconds = fog_counts.values / fs
    fog_minutes = fog_seconds / 60.0
    fog_labels = [FOG_LABELS.get(k, str(k)) for k in fog_order]

    fig, axs = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    _triptych(axs, fog_labels, fog_samples, fog_seconds, fog_minutes,
              title_prefix="FOG vs Non-FOG",
              colors=["#a6d8f0", "#f2a6a6", "#bfe8b5"])
    fog_path = os.path.join(outdir, "fog_triptych.png")
    fig.savefig(fog_path, bbox_inches="tight")
    plt.close(fig)

    # ---------- FOG severity (only during FOG) ----------
    if 'fog_severity' not in df.columns:
        raise ValueError("DataFrame must contain a 'fog_severity' column.")

    # Consider only samples where fog==1 (severity is meaningful then)
    dsev = df[df['fog'] == 1].copy()
    # Keep severities 1..3 in fixed order
    sev_order = [1, 2, 3]
    sev_counts = dsev['fog_severity'].value_counts().reindex(sev_order).fillna(0).astype(int)
    sev_samples = sev_counts.values
    sev_seconds = sev_samples / fs
    sev_minutes = sev_seconds / 60.0
    sev_labels = [FOG_SEVERITY_MAP.get(k, str(k)) for k in sev_order]

    fig, axs = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    _triptych(axs, sev_labels, sev_samples, sev_seconds, sev_minutes,
              title_prefix="FOG Severity",
              colors=["#a6d8f0", "#f2a6a6", "#bfe8b5"])
    sev_path = os.path.join(outdir, "fog_severity_triptych.png")
    fig.savefig(sev_path, bbox_inches="tight")
    plt.close(fig)

    # ---------- Summary numbers you can cite in the paper ----------
    summary = {
        "fs_hz": fs,
        "fog": {
            "samples": {label: int(s) for label, s in zip(fog_labels, fog_samples)},
            "seconds": {label: float(s) for label, s in zip(fog_labels, fog_seconds)},
            "minutes": {label: float(m) for label, m in zip(fog_labels, fog_minutes)},
            "total_samples": int(np.sum(fog_samples)),
            "total_seconds": float(np.sum(fog_seconds)),
            "total_minutes": float(np.sum(fog_minutes)),
            "share_percent": {
                label: (int(s) / max(1, int(np.sum(fog_samples)))) * 100.0
                for label, s in zip(fog_labels, fog_samples)
            }
        },
        "fog_severity": {
            "samples": {label: int(s) for label, s in zip(sev_labels, sev_samples)},
            "seconds": {label: float(s) for label, s in zip(sev_labels, sev_seconds)},
            "minutes": {label: float(m) for label, m in zip(sev_labels, sev_minutes)},
            "total_samples": int(np.sum(sev_samples)),
            "total_seconds": float(np.sum(sev_seconds)),
            "total_minutes": float(np.sum(sev_minutes)),
            "share_percent": {
                label: (int(s) / max(1, int(np.sum(sev_samples)))) * 100.0
                for label, s in zip(sev_labels, sev_samples)
            }
        }
    }

    return {
        "fog_triptych": fog_path,
        "fog_severity_triptych": sev_path,
        "summary": summary
    }


In [ ]:
# ---------------- Example ----------------
outs = make_fog_distribution_figures(fog_star, outdir="paper_figures", fs=60.0)
print(outs["summary"])

{'fs_hz': 60.0, 'fog': {'samples': {'Non-FOG': 259503, 'FOG': 64327}, 'seconds': {'Non-FOG': 4325.05, 'FOG': 1072.1166666666666}, 'minutes': {'Non-FOG': 72.08416666666668, 'FOG': 17.86861111111111}, 'total_samples': 323830, 'total_seconds': 5397.166666666667, 'total_minutes': 89.95277777777778, 'share_percent': {'Non-FOG': 80.13556495692184, 'FOG': 19.864435043078156}}, 'fog_severity': {'samples': {'Shuffling': 5996, 'Trembling': 13976, 'Akinesia': 44355}, 'seconds': {'Shuffling': 99.93333333333334, 'Trembling': 232.93333333333334, 'Akinesia': 739.25}, 'minutes': {'Shuffling': 1.6655555555555557, 'Trembling': 3.8822222222222225, 'Akinesia': 12.320833333333333}, 'total_samples': 64327, 'total_seconds': 1072.1166666666668, 'total_minutes': 17.86861111111111, 'share_percent': {'Shuffling': 9.321124877578622, 'Trembling': 21.726491208979123, 'Akinesia': 68.95238391344226}}}


In [8]:
# ======= fogstar_fog_event_durations.py =======
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compute_fog_events(df: pd.DataFrame, fs: float = 60.0) -> pd.DataFrame:
    """
    Identify contiguous FoG=1 segments, compute their duration.
    Returns DataFrame with columns: ['start_idx','end_idx','duration_samples','duration_sec']
    """
    if 'fog' not in df.columns:
        raise ValueError("DataFrame must contain a 'fog' column (0/1).")

    fog = df['fog'].to_numpy().astype(int)
    events = []
    in_event, start = False, None
    for i, val in enumerate(fog):
        if val == 1 and not in_event:
            in_event, start = True, i
        elif val == 0 and in_event:
            in_event = False
            events.append((start, i-1))
    if in_event:  # last event extends to end
        events.append((start, len(fog)-1))

    out = []
    for s, e in events:
        dur_samp = e - s + 1
        dur_sec = dur_samp / fs
        out.append((s, e, dur_samp, dur_sec))
    return pd.DataFrame(out, columns=['start_idx','end_idx','duration_samples','duration_sec'])

def plot_fog_event_duration_distribution(events_df: pd.DataFrame,
                                         outdir="paper_figures",
                                         prefix="fog_events",
                                         logx: bool = False):
    """
    Make histogram (and optionally KDE) of FoG event durations.
    Saves PNG, returns path.
    """
    os.makedirs(outdir, exist_ok=True)
    durations = events_df['duration_sec'].to_numpy()

    plt.rcParams.update({
        "font.size": 13,
        "axes.titlesize": 16,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.dpi": 150,
        "savefig.dpi": 300
    })

    fig, ax = plt.subplots(figsize=(8,5))
    bins = np.linspace(0, np.percentile(durations, 95), 30)  # robust binning up to 95th percentile
    ax.hist(durations, bins=bins, color="#a6d8f0", edgecolor="white", alpha=0.8, density=False)
    ax.set_title("Distribution of FoG Event Durations")
    ax.set_xlabel("Duration (s)")
    ax.set_ylabel("Count of Events")
    if logx:
        ax.set_xscale("log")
        ax.set_xlabel("Duration (s, log scale)")
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    plt.tight_layout()

    outpath = os.path.join(outdir, f"{prefix}_duration_hist.png")
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

    # Optional: Boxplot view
    fig, ax = plt.subplots(figsize=(6,4))
    ax.boxplot(durations, vert=False, patch_artist=True,
               boxprops=dict(facecolor="#f2a6a6", alpha=0.7))
    ax.set_title("FoG Event Duration Spread")
    ax.set_xlabel("Duration (s)")
    if logx:
        ax.set_xscale("log")
        ax.set_xlabel("Duration (s, log scale)")
    plt.tight_layout()
    outpath_box = os.path.join(outdir, f"{prefix}_duration_box.png")
    fig.savefig(outpath_box, bbox_inches="tight")
    plt.close(fig)

    # Simple stats for reporting
    summary = {
        "n_events": len(durations),
        "mean_s": float(np.mean(durations)) if len(durations) else 0,
        "median_s": float(np.median(durations)) if len(durations) else 0,
        "min_s": float(np.min(durations)) if len(durations) else 0,
        "max_s": float(np.max(durations)) if len(durations) else 0,
        "95th_percentile_s": float(np.percentile(durations, 95)) if len(durations) else 0
    }

    return {"hist": outpath, "box": outpath_box, "summary": summary}

In [9]:
events = compute_fog_events(fog_star, fs=60.0)
outs = plot_fog_event_duration_distribution(events, outdir="paper_figures", logx=False)
print(outs["summary"])

{'n_events': 101, 'mean_s': 10.615016501650166, 'median_s': 5.3, 'min_s': 0.6, 'max_s': 108.68333333333334, '95th_percentile_s': 40.2}


In [10]:
# ======= fogstar_fog_event_durations_by_severity.py =======

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FOG_SEVERITY_MAP = {
    1: "Shuffling",
    2: "Trembling",
    3: "Akinesia"
}

def compute_fog_events_by_severity(df: pd.DataFrame, fs: float = 60.0) -> pd.DataFrame:
    """
    Identify contiguous FoG=1 segments, splitting if severity changes.
    Returns DataFrame with: [start_idx, end_idx, duration_sec, severity]
    """
    if 'fog' not in df.columns or 'fog_severity' not in df.columns:
        raise ValueError("DataFrame must contain 'fog' and 'fog_severity'.")

    fog = df['fog'].to_numpy().astype(int)
    sev = df['fog_severity'].to_numpy()
    events = []
    in_event, start, start_sev = False, None, None

    for i, val in enumerate(fog):
        if val == 1 and not in_event:  # start new event
            in_event, start, start_sev = True, i, sev[i]
        elif val == 1 and in_event:
            if sev[i] != start_sev:  # severity changed -> close and restart
                events.append((start, i-1, start_sev))
                start, start_sev = i, sev[i]
        elif val == 0 and in_event:  # FoG ended
            events.append((start, i-1, start_sev))
            in_event = False
    if in_event:  # close last
        events.append((start, len(fog)-1, start_sev))

    out = []
    for s, e, sev in events:
        dur_samp = e - s + 1
        dur_sec = dur_samp / fs
        out.append((s, e, dur_sec, sev))
    return pd.DataFrame(out, columns=['start_idx','end_idx','duration_sec','severity'])

def plot_fog_event_durations_by_severity(events_df: pd.DataFrame,
                                         outdir="paper_figures",
                                         prefix="fog_events_by_severity",
                                         logx: bool = False):
    """
    Plot per-severity histograms of FoG event durations.
    One figure with 3 subplots (shuffling, trembling, akinesia).
    """
    os.makedirs(outdir, exist_ok=True)
    plt.rcParams.update({
        "font.size": 13,
        "axes.titlesize": 16,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.dpi": 150,
        "savefig.dpi": 300
    })

    fig, axs = plt.subplots(1, 3, figsize=(18,5), constrained_layout=True)
    summary = {}

    for j, sev in enumerate([1,2,3]):
        durations = events_df.query("severity == @sev")['duration_sec'].to_numpy()
        if len(durations) == 0:
            axs[j].text(0.5,0.5,"No events", ha="center", va="center")
            axs[j].set_title(FOG_SEVERITY_MAP[sev])
            axs[j].axis("off")
            summary[FOG_SEVERITY_MAP[sev]] = {}
            continue
        bins = np.linspace(0, np.percentile(durations, 95), 25)
        axs[j].hist(durations, bins=bins, color="#a6d8f0", edgecolor="white", alpha=0.8)
        axs[j].set_title(f"{FOG_SEVERITY_MAP[sev]} (n={len(durations)})")
        axs[j].set_xlabel("Duration (s)" + (" [log]" if logx else ""))
        axs[j].set_ylabel("Count")
        if logx:
            axs[j].set_xscale("log")
        axs[j].grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)

        summary[FOG_SEVERITY_MAP[sev]] = {
            "n_events": int(len(durations)),
            "mean_s": float(np.mean(durations)),
            "median_s": float(np.median(durations)),
            "min_s": float(np.min(durations)),
            "max_s": float(np.max(durations)),
            "95th_percentile_s": float(np.percentile(durations,95))
        }

    outpath = os.path.join(outdir, f"{prefix}_hist.png")
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

    return {"figure": outpath, "summary": summary}

In [11]:
# ---------------- Example ----------------
events_sev = compute_fog_events_by_severity(fog_star, fs=60.0)
outs = plot_fog_event_durations_by_severity(events_sev, outdir="paper_figures", logx=False)
outs["summary"]


{'Shuffling': {'n_events': 33,
  'mean_s': 3.0282828282828285,
  'median_s': 2.4,
  'min_s': 0.6,
  'max_s': 9.9,
  '95th_percentile_s': 7.039999999999999},
 'Trembling': {'n_events': 40,
  'mean_s': 5.823333333333333,
  'median_s': 4.9,
  'min_s': 0.2,
  'max_s': 24.1,
  '95th_percentile_s': 12.370833333333325},
 'Akinesia': {'n_events': 37,
  'mean_s': 19.979729729729733,
  'median_s': 12.4,
  'min_s': 1.8,
  'max_s': 108.68333333333334,
  '95th_percentile_s': 71.6566666666665}}